In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)

print("✅ Imports successful")

✅ Imports successful


In [2]:
import os

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
path = os.path.join(BASE_DIR, "data", "processed", "laps_cleaned.csv")

df = pd.read_csv(path)

print(f"Loaded {len(df):,} laps")
print(f"Columns: {list(df.columns)}")

Loaded 70,344 laps
Columns: ['Time_x', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason', 'FastF1Generated', 'IsAccurate', 'Year', 'RoundNumber', 'EventName', 'Time_y', 'AirTemp', 'TrackTemp', 'Humidity', 'WindSpeed', 'Rainfall', 'LapTimeSeconds', 'Sector1Seconds', 'Sector2Seconds', 'Sector3Seconds']


In [3]:
def parse_laptime(val):
    try:
        return pd.to_timedelta(val).total_seconds()
    except:
        return None

# Only re-parse if not already numeric
if df["LapTimeSeconds"].dtype == object:
    df["LapTimeSeconds"] = df["LapTime"].apply(parse_laptime)
    df["Sector1Seconds"] = df["Sector1Time"].apply(parse_laptime)
    df["Sector2Seconds"] = df["Sector2Time"].apply(parse_laptime)
    df["Sector3Seconds"] = df["Sector3Time"].apply(parse_laptime)

print("✅ Lap times confirmed numeric")
print(df["LapTimeSeconds"].describe())

✅ Lap times confirmed numeric
count   70344.000
mean       89.886
std        12.341
min        67.012
25%        81.025
50%        88.055
75%        98.023
max       149.985
Name: LapTimeSeconds, dtype: float64


In [4]:
# Ordinal encoding — reflects tyre hardness order
compound_map = {
    "SOFT": 1,
    "MEDIUM": 2,
    "HARD": 3,
    "INTERMEDIATE": 4,
    "WET": 5
}

df["CompoundEncoded"] = df["Compound"].map(compound_map)

# One-hot encoding — gives the model more flexibility
compound_dummies = pd.get_dummies(df["Compound"], prefix="Compound")
df = pd.concat([df, compound_dummies], axis=1)

print("Compound columns added:")
print([c for c in df.columns if "Compound" in c])

Compound columns added:
['Compound', 'CompoundEncoded', 'Compound_HARD', 'Compound_INTERMEDIATE', 'Compound_MEDIUM', 'Compound_SOFT', 'Compound_WET']


In [6]:
def safe_polyfit(g):
    if len(g) <= 10:
        return np.nan
    if g["TyreLife"].nunique() < 2:  # all same tyre life value → can't fit a line
        return np.nan
    try:
        return np.polyfit(g["TyreLife"], g["LapTimeSeconds"], 1)[0]
    except np.linalg.LinAlgError:
        return np.nan

deg_rate = (
    df.groupby(["EventName", "Compound"])
    .apply(safe_polyfit)
    .reset_index()
)
deg_rate.columns = ["EventName", "Compound", "DegradationRate"]

df = df.merge(deg_rate, on=["EventName", "Compound"], how="left")

print("Degradation rate sample:")
print(deg_rate.sort_values("DegradationRate", ascending=False).head(10))

Degradation rate sample:
                    EventName      Compound  DegradationRate
17         Belgian Grand Prix  INTERMEDIATE            0.423
53     Mexico City Grand Prix          SOFT            0.099
60          Monaco Grand Prix          SOFT            0.074
37  Emilia Romagna Grand Prix        MEDIUM            0.073
7         Austrian Grand Prix          HARD            0.030
70       Singapore Grand Prix          SOFT            0.025
2        Abu Dhabi Grand Prix          SOFT            0.019
30         Chinese Grand Prix        MEDIUM            0.018
72         Spanish Grand Prix        MEDIUM            0.014
39       Hungarian Grand Prix          HARD            0.012


In [7]:
# Calculate median lap time per session (year + event)
session_median = (
    df.groupby(["Year", "EventName"])["LapTimeSeconds"]
    .median()
    .reset_index()
    .rename(columns={"LapTimeSeconds": "SessionMedianLapTime"})
)

df = df.merge(session_median, on=["Year", "EventName"], how="left")

# Relative pace = how much faster/slower than the median this lap is
# Negative = faster than median, Positive = slower
df["RelativePace"] = df["LapTimeSeconds"] - df["SessionMedianLapTime"]

print("Relative pace stats:")
print(df["RelativePace"].describe())

Relative pace stats:
count   70344.000
mean        1.376
std         7.208
min       -14.890
25%        -0.896
50%         0.000
75%         0.966
max        67.483
Name: RelativePace, dtype: float64


In [8]:
# Get total laps per race
race_length = (
    df.groupby(["Year", "EventName"])["LapNumber"]
    .max()
    .reset_index()
    .rename(columns={"LapNumber": "TotalLaps"})
)

df = df.merge(race_length, on=["Year", "EventName"], how="left")

# Track evolution = how far through the race we are (0 = start, 1 = end)
# Earlier laps have more track evolution effect (more rubber laid down over time)
df["TrackEvolution"] = df["LapNumber"] / df["TotalLaps"]

print("Track evolution stats:")
print(df["TrackEvolution"].describe())

Track evolution stats:
count   70344.000
mean        0.514
std         0.284
min         0.026
25%         0.268
50%         0.514
75%         0.761
max         1.000
Name: TrackEvolution, dtype: float64


In [9]:
# Estimated remaining fuel as a fraction of starting fuel
# At lap 1: ~1.0 (full tank), at final lap: ~0.0 (nearly empty)
df["FuelLoad"] = 1 - (df["LapNumber"] / df["TotalLaps"])

print("Fuel load stats:")
print(df["FuelLoad"].describe())

Fuel load stats:
count   70344.000
mean        0.486
std         0.284
min         0.000
25%         0.239
50%         0.486
75%         0.732
max         0.974
Name: FuelLoad, dtype: float64


In [17]:
team_pace = (
    df.groupby(["Year", "Team"])["RelativePace"]
    .mean()
    .reset_index()
    .rename(columns={"RelativePace": "TeamPerformanceIndex"})
)
df = df.drop(columns=["TeamPerformanceIndex"], errors="ignore")
df = df.merge(team_pace, on=["Year", "Team"], how="left")

print("Team performance index (lower = faster):")
print(
    team_pace[team_pace["Year"] == 2025]
    .sort_values("TeamPerformanceIndex")
    .to_string(index=False)
)

Team performance index (lower = faster):
 Year            Team  TeamPerformanceIndex
 2025         McLaren                 0.734
 2025         Ferrari                 0.972
 2025        Mercedes                 1.240
 2025 Red Bull Racing                 1.248
 2025        Williams                 1.538
 2025    Racing Bulls                 1.698
 2025    Haas F1 Team                 1.803
 2025    Aston Martin                 1.845
 2025     Kick Sauber                 1.935
 2025          Alpine                 1.998


In [ ]:
if "TrackTemp" in df.columns:
    # Normalize track temp to 0-1 range
    df["TrackTempNorm"] = (
        (df["TrackTemp"] - df["TrackTemp"].min()) /
        (df["TrackTemp"].max() - df["TrackTemp"].min())
    )

    # Interaction: soft tyres are more sensitive to heat than hard tyres
    df["TempCompoundInteraction"] = df["TrackTempNorm"] * df["CompoundEncoded"]

    print("Weather interaction feature added")
    print(df[["TrackTemp", "TrackTempNorm", "CompoundEncoded",
              "TempCompoundInteraction"]].head(50))
else:
    print("⚠️ TrackTemp not available — skipping weather interaction")
    df["TrackTempNorm"] = 0
    df["TempCompoundInteraction"] = 0

Weather interaction feature added
    TrackTemp  TrackTempNorm  CompoundEncoded  TempCompoundInteraction
0      31.100          0.400                1                    0.400
1      31.100          0.400                1                    0.400
2      31.100          0.400                1                    0.400
3      31.100          0.400                1                    0.400
4      31.100          0.400                1                    0.400
5      31.100          0.400                1                    0.400
6      31.100          0.400                1                    0.400
7      31.100          0.400                1                    0.400
8      31.100          0.400                1                    0.400
9      31.100          0.400                1                    0.400
10     31.100          0.400                1                    0.400
11     31.100          0.400                1                    0.400
12     31.100          0.400               

: 

In [19]:
# Create a consistent track ID mapping
tracks = sorted(df["EventName"].unique())
track_map = {track: idx for idx, track in enumerate(tracks)}

df["TrackEncoded"] = df["EventName"].map(track_map)

# Save the mapping so we can use it in the app later
import json
import os

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
mapping_path = os.path.join(BASE_DIR, "data", "processed", "track_mapping.json")

with open(mapping_path, "w") as f:
    json.dump(track_map, f, indent=2)

print(f"✅ Track mapping saved — {len(track_map)} tracks")
print(track_map)

✅ Track mapping saved — 24 tracks
{'Abu Dhabi Grand Prix': 0, 'Australian Grand Prix': 1, 'Austrian Grand Prix': 2, 'Azerbaijan Grand Prix': 3, 'Bahrain Grand Prix': 4, 'Belgian Grand Prix': 5, 'British Grand Prix': 6, 'Canadian Grand Prix': 7, 'Chinese Grand Prix': 8, 'Dutch Grand Prix': 9, 'Emilia Romagna Grand Prix': 10, 'Hungarian Grand Prix': 11, 'Italian Grand Prix': 12, 'Japanese Grand Prix': 13, 'Las Vegas Grand Prix': 14, 'Mexico City Grand Prix': 15, 'Miami Grand Prix': 16, 'Monaco Grand Prix': 17, 'Qatar Grand Prix': 18, 'Saudi Arabian Grand Prix': 19, 'Singapore Grand Prix': 20, 'Spanish Grand Prix': 21, 'São Paulo Grand Prix': 22, 'United States Grand Prix': 23}


In [20]:
drivers = sorted(df["Driver"].unique())
driver_map = {driver: idx for idx, driver in enumerate(drivers)}
df["DriverEncoded"] = df["Driver"].map(driver_map)

# Save driver mapping
driver_mapping_path = os.path.join(BASE_DIR, "data", "processed", "driver_mapping.json")
with open(driver_mapping_path, "w") as f:
    json.dump(driver_map, f, indent=2)

print(f"✅ Driver mapping saved — {len(driver_map)} drivers")

✅ Driver mapping saved — 28 drivers


In [21]:
# These are the features the model will train on
FEATURE_COLUMNS = [
    # Tyre features (most important)
    "TyreLife",
    "CompoundEncoded",
    "Compound_SOFT",
    "Compound_MEDIUM", 
    "Compound_HARD",
    "DegradationRate",

    # Race context
    "LapNumber",
    "FuelLoad",
    "TrackEvolution",

    # Car and driver
    "TeamPerformanceIndex",
    "DriverEncoded",
    "TrackEncoded",

    # Weather
    "TrackTempNorm",
    "TempCompoundInteraction",

    # Target variable
    "LapTimeSeconds",

    # Keep these for reference but don't use as features
    "Year",
    "EventName",
    "Driver",
    "Team",
    "Compound"
]

# Only keep columns that actually exist
available = [c for c in FEATURE_COLUMNS if c in df.columns]
missing_cols = [c for c in FEATURE_COLUMNS if c not in df.columns]

if missing_cols:
    print(f"⚠️  These columns were not found and will be skipped: {missing_cols}")

df_features = df[available].copy()

# Drop any rows where critical features are missing
critical = ["TyreLife", "LapTimeSeconds", "CompoundEncoded", "TeamPerformanceIndex"]
critical_available = [c for c in critical if c in df_features.columns]
df_features = df_features.dropna(subset=critical_available)

print(f"\n✅ Final feature dataframe shape: {df_features.shape}")
print(f"Features for model: {[c for c in available if c not in ['Year', 'EventName', 'Driver', 'Team', 'Compound', 'LapTimeSeconds']]}")


✅ Final feature dataframe shape: (70324, 20)
Features for model: ['TyreLife', 'CompoundEncoded', 'Compound_SOFT', 'Compound_MEDIUM', 'Compound_HARD', 'DegradationRate', 'LapNumber', 'FuelLoad', 'TrackEvolution', 'TeamPerformanceIndex', 'DriverEncoded', 'TrackEncoded', 'TrackTempNorm', 'TempCompoundInteraction']


In [22]:
numeric_features = [
    "TyreLife", "CompoundEncoded", "DegradationRate",
    "LapNumber", "FuelLoad", "TrackEvolution",
    "TeamPerformanceIndex", "TrackTempNorm", "TempCompoundInteraction"
]

available_numeric = [c for c in numeric_features if c in df_features.columns]

correlations = (
    df_features[available_numeric + ["LapTimeSeconds"]]
    .corr()["LapTimeSeconds"]
    .drop("LapTimeSeconds")
    .sort_values()
)

fig = px.bar(
    x=correlations.values,
    y=correlations.index,
    orientation="h",
    title="Feature Correlation with Lap Time",
    labels={"x": "Correlation coefficient", "y": "Feature"},
    color=correlations.values,
    color_continuous_scale="RdBu_r"
)

fig.update_layout(template="plotly_dark", height=500, showlegend=False)
fig.show()

In [23]:
output_path = os.path.join(BASE_DIR, "data", "processed", "laps_features.csv")
df_features.to_csv(output_path, index=False)

print(f"✅ Feature dataset saved to {output_path}")
print(f"   Shape: {df_features.shape}")
print(f"   Columns: {list(df_features.columns)}")

✅ Feature dataset saved to c:\yashas\f1-strategy-optimizer\data\processed\laps_features.csv
   Shape: (70324, 20)
   Columns: ['TyreLife', 'CompoundEncoded', 'Compound_SOFT', 'Compound_MEDIUM', 'Compound_HARD', 'DegradationRate', 'LapNumber', 'FuelLoad', 'TrackEvolution', 'TeamPerformanceIndex', 'DriverEncoded', 'TrackEncoded', 'TrackTempNorm', 'TempCompoundInteraction', 'LapTimeSeconds', 'Year', 'EventName', 'Driver', 'Team', 'Compound']
